# Calculating the Allan Variance for our dataset.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [48]:
DATA_LOC = '../../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(os.path.exists(FINAL_DOC))

True


In [ ]:
# Allan Var doc
allan_var = 'allan_var-difs-multiling.parquet'

#### Load data

In [9]:
# df = pd.read_table(FINAL_DOC, sep='\t')
df = pd.read_parquet(FINAL_DOC)
df.shape

(14065648, 12)

In [15]:
df = df.rename(columns={'x_turn_id': 'groups'})

In [16]:
df['conversation_id'].nunique()

1285

In [17]:
df.head()

,Hxy,nx,ny,turn_delta,self,groups,x_speaker,corpus,conversation_id,lang,sample_delta,resid
0,2.765743,12.0,10.0,2.0,False,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,1,-0.159806
1,2.836436,12.0,7.0,6.0,False,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,2,-0.141870
2,2.906804,12.0,8.0,9.0,False,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,3,-0.053917
3,3.063219,12.0,5.0,10.0,False,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,4,0.049741
4,2.971834,12.0,9.0,12.0,False,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,5,0.028699


In [18]:
df.shape

(14065648, 12)

### Merge on Self vs. Other

In [19]:
merge_columns = ['groups', 'sample_delta']
value_columns = ['Hxy']

In [20]:
df = pd.merge(
    left=df.loc[df['self']], left_on=merge_columns,
    right=df[merge_columns+value_columns].loc[~df['self']], right_on=merge_columns,
    how='left'
)
df.shape

(8021922, 13)

In [21]:
df.isna().sum()

Hxy_x                    0
nx                       0
ny                       0
turn_delta               0
self                     0
groups                   0
x_speaker                0
corpus                   0
conversation_id          0
lang                     0
sample_delta             0
resid                    0
Hxy_y              2741173
dtype: int64

In [22]:
df['Hxy'] = df['Hxy_y'] - df['Hxy_x']

In [23]:
del df['Hxy_x']
del df['Hxy_y']

In [24]:
df.head()

,nx,ny,turn_delta,self,groups,x_speaker,corpus,conversation_id,lang,sample_delta,resid,Hxy
0,12.0,7.0,3.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,1,0.102238,-0.314801
1,12.0,8.0,7.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,2,0.033382,-0.157667
2,12.0,12.0,8.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,3,0.090274,-0.073849
3,12.0,5.0,11.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,4,0.049741,0.000000
4,12.0,9.0,16.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,5,0.000002,0.028697


In [25]:
print(df.shape)
df = df.loc[~df['Hxy'].isna()]
df.shape

(8021922, 12)


(5280749, 12)

### Merge `copied doc` Method.

In [26]:
merge_columns = ['groups', 'sample_delta']

In [27]:
value_columns = ['Hxy']

#### First turn after the current turn

In [28]:
# make a copy of our df
df_ = df.copy()

# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [29]:
# merge the copied doc with the actual doc using dyad ID, self, 
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

(5280749, 13)


nx                      0
ny                      0
turn_delta              0
self                    0
groups                  0
x_speaker               0
corpus                  0
conversation_id         0
lang                    0
sample_delta            0
resid                   0
Hxy_x                   0
Hxy_y              299963
dtype: int64

#### Second turn after the current turn

In [30]:
# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [31]:
# merge the copied doc with the actual doc using dyad ID, self, 
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

(5280749, 14)


nx                      0
ny                      0
turn_delta              0
self                    0
groups                  0
x_speaker               0
corpus                  0
conversation_id         0
lang                    0
sample_delta            0
resid                   0
Hxy_x                   0
Hxy_y              299963
Hxy                595253
dtype: int64

#### Final checks

In [32]:
# check what row x column combos have NaNs
df.isna().sum()

nx                      0
ny                      0
turn_delta              0
self                    0
groups                  0
x_speaker               0
corpus                  0
conversation_id         0
lang                    0
sample_delta            0
resid                   0
Hxy_x                   0
Hxy_y              299963
Hxy                595253
dtype: int64

In [33]:
# get a sample of the NaN containing rows
df[['groups', 'sample_delta', 'Hxy_x', 'Hxy_y', 'Hxy']].loc[df['Hxy'].isna()].sample(n=10).head(n=10)

,groups,sample_delta,Hxy_x,Hxy_y,Hxy
918293,croation-cro-croation-cro-2015_4_1-326,12,-0.064913,NaN,NaN
5109796,callfriend-ko-6559-191,22,-0.289478,NaN,NaN
3982832,callhome-spa-callhome-spa-0899-98,19,-0.224614,-0.043756,NaN
4782880,callhome-zho-callhome-zho-1582-81,23,0.300201,NaN,NaN
3106632,callhome-deu-callhome-deu-5159-55,24,-0.134413,NaN,NaN
4056912,callhome-spa-callhome-spa-1100-14,26,-0.449142,NaN,NaN
3761893,callhome-eng-callhome-eng-6161-354,7,-0.176226,0.049959,NaN
1701882,callfriend-fra-q-callfriend-fra-q-5474-699,20,0.022717,0.101147,NaN
3805836,callhome-eng-callhome-eng-6267-702,22,0.075821,0.078422,NaN
1729374,callfriend-fra-q-callfriend-fra-q-5480-622,21,-0.100757,NaN,NaN


In [34]:
df = df.loc[~df['Hxy'].isna()]
df.shape

(4685496, 14)

In [35]:
df = df.rename(columns={'Hxy_x': 'resid', 'Hxy_y': 'resid_1', 'Hxy': 'resid_2', 'resid': 'residual'})

In [36]:
df[['groups', 'resid', 'resid_1', 'resid_2']].head()

,groups,resid,resid_1,resid_2
0,croation-cro-croation-cro-2011_56-5,-0.314801,-0.157667,-0.073849
1,croation-cro-croation-cro-2011_56-5,-0.157667,-0.073849,0.000000
2,croation-cro-croation-cro-2011_56-5,-0.073849,0.000000,0.028697
3,croation-cro-croation-cro-2011_56-5,0.000000,0.028697,-0.037892
4,croation-cro-croation-cro-2011_56-5,0.028697,-0.037892,-0.231420


### Create Allan Variance Scale Parameter

In [37]:
group_cols = ['groups', 'self']
number_cols = ['sample_delta']

In [38]:
N = df[group_cols+number_cols].groupby(by=group_cols).agg('count').reset_index()
N = N.rename(columns={'sample_delta': 'N'})

In [39]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=N, right_on=group_cols,
    how='left'
)
df.shape

(4685496, 15)

In [40]:
tau = df[group_cols+number_cols].groupby(by=group_cols).agg('max').reset_index()
tau = tau.rename(columns={'sample_delta': 'tau'})

In [41]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=tau, right_on=group_cols,
    how='left'
)
df.shape

(4685496, 16)

In [42]:
df.isna().sum()

nx                 0
ny                 0
turn_delta         0
self               0
groups             0
x_speaker          0
corpus             0
conversation_id    0
lang               0
sample_delta       0
residual           0
resid              0
resid_1            0
resid_2            0
N                  0
tau                0
dtype: int64

In [43]:
df.head()

,nx,ny,turn_delta,self,groups,x_speaker,corpus,conversation_id,lang,sample_delta,residual,resid,resid_1,resid_2,N,tau
0,12.0,7.0,3.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,1,0.102238,-0.314801,-0.157667,-0.073849,21,21
1,12.0,8.0,7.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,2,0.033382,-0.157667,-0.073849,0.000000,21,21
2,12.0,12.0,8.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,3,0.090274,-0.073849,0.000000,0.028697,21,21
3,12.0,5.0,11.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,4,0.049741,0.000000,0.028697,-0.037892,21,21
4,12.0,9.0,16.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,5,0.000002,0.028697,-0.037892,-0.231420,21,21


## Calculate Allan $\sigma^2$

In [44]:
df = df.loc[~df['resid_2'].isna()]

In [45]:
df['allan_var'] = (1/(2*(df['tau']**2))) * ((df['resid_2'] - (2*df['resid_1']) + df['resid'])**2)

In [46]:
df.head()

,nx,ny,turn_delta,self,groups,x_speaker,corpus,conversation_id,lang,sample_delta,residual,resid,resid_1,resid_2,N,tau,allan_var
0,12.0,7.0,3.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,1,0.102238,-0.314801,-0.157667,-0.073849,21,21,6.094507e-06
1,12.0,8.0,7.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,2,0.033382,-0.157667,-0.073849,0.000000,21,21,1.126606e-07
2,12.0,12.0,8.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,3,0.090274,-0.073849,0.000000,0.028697,21,21,2.311523e-06
3,12.0,5.0,11.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,4,0.049741,0.000000,0.028697,-0.037892,21,21,1.029395e-05
4,12.0,9.0,16.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,4,5,0.000002,0.028697,-0.037892,-0.231420,21,21,1.826944e-05


### Save outputs

In [49]:
# df.to_csv(
#     allan_var,
#     index=False,
#     encoding='utf-8',
#     sep='\t'
# )

In [ ]:
df.to_parquet(allan_var)